# SpaPath OSCC workflow

This notebook follows one linear workflow: load two single-cell references and the GSM6339632_s2 disease data, attach image embeddings when needed, preprocess and build graphs, learn initial embeddings, cluster, integrate, detect pathological regions, construct the all-gene disease dataset, and evaluate TC-to-LE enrichment against ground-truth tissue labels.

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import scanpy as sc
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import spapath_model
import spapath_utils

warnings.filterwarnings("ignore")

In [ ]:
DATASET_ID = "OSCC"
REFERENCE_SECTIONS = ["GM241", "BM169"]
DISEASE_SECTION = "GSM6339632_s2"
DEVICE = "cuda"
SEED = 123

DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_ROOT / DATASET_ID
OUTPUT_DIR = PROJECT_ROOT / "outputs" / DATASET_ID
FIGURE_DIR = OUTPUT_DIR / "fig"
TC_LE_GENE_FILE = PROJECT_ROOT / "notebook" / "TC_LE_genes.csv"
HE_IMAGE_PATH = (
    PROJECT_ROOT
    / "data"
    / DATASET_ID
    / "GSM6339632_s2_tissue_hires_image.png"
)

for output_path in (OUTPUT_DIR, FIGURE_DIR):
    spapath_utils.create_dir(output_path)

adata_type_map = {
    REFERENCE_SECTIONS[0]: "sc",
    REFERENCE_SECTIONS[1]: "sc",
    DISEASE_SECTION: "ST_with_HE",
}
sections = list(adata_type_map)

CELLTYPE_PALETTE = {
    "core": "#E09F3E",
    "transitory": "#C8553D",
    "edge": "#8E3B46",
    "normal": "#5C7A72",
    "nc": "#5C7A72",
}

REGION_PALETTE = {
    "Pathological regions": "#B6473F",
    "Healthy-like regions": "#6DBBD1",
}

## 1. Read reference and disease data

In [ ]:
reference_adatas = [
    sc.read_h5ad(PROCESSED_DIR / f"{section}.h5ad")
    for section in REFERENCE_SECTIONS
]
disease_adata = sc.read_h5ad(PROCESSED_DIR / f"{DISEASE_SECTION}.h5ad")

## 2. Attach image embeddings when needed

In [ ]:
if "image_embedding" not in disease_adata.obsm:
    embedding_path = (
        PROCESSED_DIR
        / "spatial"
        / f"{DISEASE_SECTION}_image_embeddings.csv"
    )
    spapath_utils.attach_image_embedding(disease_adata, embedding_path)

image_embedding_dim = disease_adata.obsm["image_embedding"].shape[1]
for reference_adata in reference_adatas:
    reference_adata.obsm["image_embedding"] = np.zeros(
        (reference_adata.n_obs, image_embedding_dim),
        dtype=np.float32,
    )

## 3. Preprocess data and build graphs

In [ ]:
batch_list = [*reference_adatas, disease_adata]
adata_full, disease_adata_all_genes = spapath_utils.preprocess(
    adata_list=batch_list,
    adata_type_map=adata_type_map,
    full_num_hvgs=3000,
    min_genes_qc=10,
    min_cells_qc=10,
)

adata_full = spapath_utils.build_graph_GAT_plus(
    adata_full=adata_full,
    adata_type_map=adata_type_map,
    K=8,
    img_threshold=0.0,
)

## 4. Learn initial embeddings

In [ ]:
model = spapath_model.Model(
    adata_full=adata_full,
    adata_type_map=adata_type_map,
    lr_pre=1e-4,
    lr=1e-4,
    n_pre_training_steps=500,
    n_training_steps=300,
    device=DEVICE,
    seed=SEED,
)

adata_full = model.initial_embedding()

## 5. Cluster observations

In [ ]:
adata_full = model.clustering(
    init_res=1.5,
    intopk=40,
)

## 6. Integrate reference and disease data

In [ ]:
adata_full = model.integrate(topk=40)

## 7. Detect pathological regions

In [ ]:
adata_full = spapath_utils.detection(
    adata=adata_full,
    embed="cell_embed",
    section_ids=sections,
    label_core="Pathological regions",
    label_other="Healthy-like regions",
    core_types=["core", "edge", "transitory"],
    celltype_key="CellType",
    batch_key="batch",
    seed=SEED,
    neighbors=30,
    threshold=0.05,
    strategy="individual",
)

display(adata_full.uns["result"])

In [ ]:
detection_figure = spapath_utils.plot_detection_umap(
    adata=adata_full,
    embed="cell_embed",
    section_id=DISEASE_SECTION,
    batch_key="batch",
    label_key="pred_label",
    label_palette=REGION_PALETTE,
    celltype_key="CellType",
    celltype_palette=CELLTYPE_PALETTE,
    seed=SEED,
    point_size=18,
    save=FIGURE_DIR / f"{DISEASE_SECTION}_detection_umap.png",
)

he_prediction_figure = spapath_utils.plot_prediction_on_he(
    adata=adata_full,
    image_path=HE_IMAGE_PATH,
    section_id=DISEASE_SECTION,
    batch_key="batch",
    label_key="pred_label",
    spatial_key="spatial",
    label_palette=REGION_PALETTE,
    coordinate_scale=0.08828852,
    point_size=8,
    save=FIGURE_DIR / f"{DISEASE_SECTION}_prediction_on_he.png",
)

## 8. Build the all-gene disease dataset

In [ ]:
disease_data = spapath_utils.build_disease_data(
    adata_full=adata_full,
    disease_adata_all_genes=disease_adata_all_genes,
    disease_section=DISEASE_SECTION,
)

## 9. Calculate TC-to-LE enrichment

In [ ]:
tc_le_results = spapath_utils.calculate_tc_le_enrichment(
    adata=disease_data,
    gene_table=TC_LE_GENE_FILE,
    label_key="pred_label",
    pathological_label="Pathological regions",
    dist_key="cell_gene_dist",
    n_permutations=5000,
    min_genes=5,
    seed=SEED,
)

## 10. Plot TC-to-LE enrichment on H&E

In [ ]:
tc_le_he_figure = spapath_utils.plot_score_on_he(
    adata=disease_data,
    image_path=HE_IMAGE_PATH,
    score_key="TC_to_LE_score",
    label_key="pred_label",
    focus_label="Pathological regions",
    spatial_key="spatial",
    coordinate_scale=0.08828852,
    background_color=REGION_PALETTE["Healthy-like regions"],
    cmap="Reds",
    point_scale=0.85,
    figsize=(3, 3),
    save=FIGURE_DIR / f"{DISEASE_SECTION}_TC_to_LE_score_on_he.png",
)

## 11. Compare enrichment across ground-truth tissue classes

In [ ]:
tc_le_figure, tc_le_statistics = spapath_utils.plot_tc_le_violin(
    adata=disease_data,
    score_key="TC_to_LE_score",
    group_key="CellType",
    predicted_label_key="pred_label",
    truth_label_key="truth_label",
    pathological_label="Pathological regions",
    group_order=["core", "transitory", "edge"],
    palette=CELLTYPE_PALETTE,
    save=FIGURE_DIR / f"{DISEASE_SECTION}_TC_to_LE_violin.png",
)